# 记忆

上一章，我们通过智能体类建立了消息循环，可以和大模型持续交流。但是在最后的测试，大模型无法在后续的交流中记住之前的交流内容（小明的名字）。原因是大模型的 API 是**无状态**的：每次调用都是独立的，我们只发了当前这一句提问，它当然不知道之前发生过什么。

解决办法也不稀奇：**每次调用时，把之前的交流记录一起发过去**。

**大模型本身并没有记忆，记忆是我们的智能体程序帮它保存的**。我们在网页上用的聊天机器人，背后也是这样做的。

In [1]:
import os
from abc import ABC, abstractmethod
from enum import Enum
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

## 消息类

### 角色

In [2]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

### 抽象类

In [3]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

### 系统消息类

In [4]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

### 用户消息类

In [5]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

### 助手消息类

In [6]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [7]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

## 记忆类

我们用一个记忆类 `Memory` 来保存聊天记录。它很简单，就是把消息按顺序放在一个列表 `messages` 里：

- `add`：把一条新消息加到列表最后。
- `reset`：清空所有记录，相当于“失忆”。
- `history`：把所有消息拼成一段文字，方便我们查看记忆里都有什么。

上一章设计的消息类，在这里派上了用场：不管是小明说的话还是助手的回答，都是 `Message`，可以都放进同一个列表里。

In [8]:
class Memory:

    def __init__(self):
        self.messages: List[Message] = []

    def add(self, message: Message):
        self.messages.append(message)

    def reset(self):
        self.messages = []

    def history(self) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in self.messages
        )

## 大模型调用器

### 基础类

In [9]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

### OpenAI 调用器

In [10]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

### Claude 调用器

In [11]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

## 智能体类

为了让大模型拥有记忆，`Agent` 里也改动了两处：

1. `__init__` 里多了一个 `self.memory`，每个 Agent 都有自己的记忆；
2. 调用大模型时，发送的不再只是当前这句话，而是 `self.memory.messages + [user_message]`，也就是“之前的全部记录 + 当前的问题”；拿到回答以后，再把问题和回答都存进记忆。

这里要注意顺序：先调用大模型，**成功以后**再把问题和回答存进去。如果调用出错了（比如网络断了），记忆里就不会留下一个没有回答的问题。

In [12]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage):
        self.provider = provider
        self.system = system
        self.memory = Memory()

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            user_message = UserMessage(message)
            response = self.provider.call(system or self.system, self.memory.messages + [user_message])
            self.memory.add(user_message)
            self.memory.add(response)
            print(f"AI: {response.summary()}")

## 大模型调用

上一章同样的问题，我们再问一遍：

In [13]:
provider = OpenAIProvider()
agent = Agent(provider, system_message)
agent.chat(["你好，我叫小明，正在学这本书。", "我叫什么名字？"])


YOU: 你好，我叫小明，正在学这本书。
AI: 你好，小明，很高兴一起学习《从大模型到智能体》！

我可以帮你：
- 讲解书里的概念，比如 LLM、Agent、工具调用、RAG、规划与记忆
- 把复杂内容用通俗的话解释
- 帮你梳理每一章的重点
- 带你做简单的 Agent 设计和代码思路分析
- 回答你学习中遇到的具体问题

你可以直接告诉我：
1. 你现在在看哪一章  
2. 哪个概念没看懂  
3. 想不想我帮你做一份学习笔记或思维导图

如果你愿意，我们可以先从“什么是智能体（Agent）”开始。

YOU: 我叫什么名字？
AI: 你叫小明。

YOU: /quit


这次学习助手答出了小明的名字。我们来看看它的记忆里都存了什么：

In [14]:
print(agent.memory.history())

[User]: 你好，我叫小明，正在学这本书。
[Assistant]: 你好，小明，很高兴一起学习《从大模型到智能体》！

我可以帮你：
- 讲解书里的概念，比如 LLM、Agent、工具调用、RAG、规划与记忆
- 把复杂内容用通俗的话解释
- 帮你梳理每一章的重点
- 带你做简单的 Agent 设计和代码思路分析
- 回答你学习中遇到的具体问题

你可以直接告诉我：
1. 你现在在看哪一章  
2. 哪个概念没看懂  
3. 想不想我帮你做一份学习笔记或思维导图

如果你愿意，我们可以先从“什么是智能体（Agent）”开始。
[User]: 我叫什么名字？
[Assistant]: 你叫小明。


每一轮的问题和回答都按顺序存了下来，下一轮会和新的问题一起发给大模型。

这时如果我们清空记忆，智能体就又“失忆”了：

In [15]:
agent.memory.reset()
agent.chat(["我叫什么名字？"])


YOU: 我叫什么名字？
AI: 我目前不知道你的名字。  
如果你愿意，可以告诉我，我就可以这样称呼你。

YOU: /quit


## 记忆的代价

把聊天记录一起发过去，问题解决了，但也带来了新的麻烦：**聊得越久，每次发送的内容就越多**。

这会带来三个问题：

- **越来越贵**。API 按 token 收费，之前的记录每一轮都要重新发一遍，每一轮都要重新付一次钱；
- **越来越慢**。要处理的内容越多，模型回答得就越慢；
- **总有装不下的一天**。每个模型一次能读入的内容都有上限，叫作**上下文窗口**（context window）。聊天记录一旦超过这个上限，API 就会直接报错。

下一章，我们来解决这个问题。

## 动手试试

- 连续问几个前后有关联的问题，比如先问“什么是变量？”，再问“能举个例子吗？”，看看学习助手能不能接上话。
- 给 `chat` 加一个 `/reset` 命令：输入 `/reset` 时清空记忆，并提示“记忆已清空”。提示：模仿 `/quit` 的写法，调用 `self.memory.reset()`。

## 小结

- 大模型本身没有记忆。要让它“记住”之前的对话，就要在每次调用时，把聊天记录一起发过去；
- 记忆类用一个列表按顺序保存消息；智能体每次调用时发送“记忆 + 当前问题”，成功后再把问题和回答存进记忆；
- 记忆越长，每次发送的内容越多：更贵、更慢，最终会超出模型的**上下文窗口**。

下一章，我们教智能体做总结：把太长的聊天记录压缩成一段摘要。